In [ ]:
import json
from IPython.display import display
import ipywidgets as widgets

data_loc = "../data/synthetic/components"

In [ ]:
with open(f'{data_loc}/motherboards.json', 'r') as file:
    mobo_data = json.load(file)

mobo_choices = [(f"{mobo['brand']} {mobo['model']}", mobo['id']) for mobo in mobo_data]

In [ ]:
CATEGORIES = ['cases', 'coolers', 'cpus', 'fans', 'gpus', 'lighting', 'motherboards', 'psus', 'ram', 'storage']
CATALOG = {}
for category in CATEGORIES:
    with open(f'{data_loc}/{category}.json', 'r') as file:
        cat_data = json.load(file)
    CATALOG[category] = cat_data

## component-specific specs layouts

- each function builds a widget containing the specs for one type of component 
- since each component type has different specs, keeping these functions separate makes my life easier to throw component specs into the main card layout

### case

In [ ]:
def get_case_specs(case):
    motherboard_sizes = ', '.join(case['motherboard_form_factors'])
    psu_sizes = ', '.join(case['supported_psu_form_factors'])
    radiator_sizes = ', '.join(f'{size} mm' for size in case['radiator_sizes_mm']) or 'None'
    drive_bays = f"2.5 in: {case['sata_bays_2_5']} | 3.5 in: {case['sata_bays_3_5']}"

    # TODO: might be better to make a normalizer function for this, but not enough data rn to justify 
    usb_header_names = {
        'USB3_19PIN': 'USB 3.0 (19-pin)',
        'USBC_20PIN': 'USB-C (20-pin)',
        'USB2_9PIN': 'USB 2.0 (9-pin)',
    }
    front_usb_headers = ', '.join(usb_header_names[header] for header in case['front_usb_headers']) or 'None'

    spec_rows = [
        ('Motherboards:', motherboard_sizes),
        ('PSU:', psu_sizes),
        ('GPU max length:', f"{case['gpu_max_length_mm']} mm"),
        ('CPU cooler max height:', f"{case['cpu_cooler_max_height_mm']} mm"),
        ('Radiators:', radiator_sizes),
        ('Drive bays:', drive_bays),
        ('Front USB:', front_usb_headers),
    ]

    return spec_rows

### cooler

In [ ]:
def get_cooler_specs(cooler):
    return widgets.VBox()

### cpu

In [ ]:
def get_cpu_specs(cpu):
    return widgets.VBox()

### fans

In [ ]:
def get_fans_specs(fans):
    return widgets.VBox()

### gpu

In [ ]:
def get_gpu_specs(gpu):
    return widgets.VBox()

### lighting

In [ ]:
def get_lighting_specs(lighting):
    return widgets.VBox()

### mobo

In [ ]:
def get_mobo_specs(mobo):
    connectors = mobo['connector_layout']['connectors']
    pcie_slots = [c['interface'].replace('PCIE_X', 'x') for c in connectors if c['kind'] == 'pcie_slot']
    usb_headers = [c['interface'] for c in connectors if c['kind'] == 'usb_header']
    rgb_headers = [c['interface'] for c in connectors if c['kind'] == 'rgb_header']
    m2_slots = [c for c in connectors if c['kind'] == 'm2_slot']
    sata_ports = [c for c in connectors if c['kind'] == 'sata_port']
    fan_headers = [c for c in connectors if c['kind'] == 'fan_header']
    power_connectors = [
        c['interface'].replace('_', ' ').replace('PIN', '-pin') 
        for c in connectors if c['kind'] == 'power_header'
    ]

    spec_rows = [
        ('Chipset:', mobo['chipset']),
        ('Socket:', mobo['cpu_socket']),
        ('Size:', mobo['form_factor']),
        ('Board:', f"{mobo['dimensions_mm']['width']} x {mobo['dimensions_mm']['height']} mm"),
        ('Memory:', f"{mobo['memory']['type']} {mobo['memory']['form_factor']}"),
        ('Slots:', str(mobo['memory']['slots'])),
        ('Max memory:', f"{mobo['memory']['max_capacity_gb']} GB"),
        ('PCIe:', ', '.join(pcie_slots)),
        ('M.2 slots:', str(len(m2_slots))),
        ('SATA ports:', str(len(sata_ports))),
        ('Headers:', f"fan {len(fan_headers)} | USB {len(usb_headers)} | RGB {len(rgb_headers)}"),
        ('Power:', ' + '.join(power_connectors)),
    ]

    return spec_rows

### psu

In [ ]:
def get_psu_specs(psu):
    return widgets.VBox()

### ram

In [ ]:
def get_ram_specs(ram):
    return widgets.VBox()

### storage

In [ ]:
def get_storage_specs(storage):
    return widgets.VBox()

## CARD MAKER

In [ ]:
def make_card(name, price, image, specs_rows):
    # CARD HEADER = |name  price|
    header = widgets.HBox(
        [
            widgets.Label(name, style={'font_weight': 'bold'}), 
            widgets.Label(price, style={'text_color': 'green'})
        ], 
        layout=widgets.Layout(width='300px', justify_content='space-between')
    )

    specs_layout = specs_widget(specs_rows)

    card = widgets.VBox(
        [header, image, specs_layout],
        layout=widgets.Layout(
            width='350px',
            height='460px',
            padding='6px',
            border='1px solid #777',
        ),
    )

    return card

# make the specs widget withthe given rows (list of tuples)
def specs_widget(spec_rows):
    specs_content = widgets.VBox([
        widgets.HBox([
            widgets.Label(label, layout=widgets.Layout(width='37%'), style={'font_weight': 'bold'}),
            widgets.Box(
                [widgets.Label(value)],
                layout=widgets.Layout(width='55%', justify_content='flex-end'),
            ),
        ], layout=widgets.Layout(width='99%'))
        for label, value in spec_rows
    ], layout=widgets.Layout(gap='1%', width='350px', min_width='350px'))

    return widgets.Box(
        [specs_content],
        layout=widgets.Layout(width='99%', height='50%', overflow='auto', display='block'),
    )


## testing

In [ ]:
# PLACEHOLDER IMAGE
# will eventually have cute lil pics for components
# for now, this:
image_placeholder = widgets.Box(
    [widgets.Label('IMAGE PLACEHOLDER')],
    layout=widgets.Layout(
        width='98%',
        height='40%',
        justify_content='center',
        align_items='center',
        border='1px solid #ccc',
    ),
)

In [ ]:
test_mobo = next(board for board in mobo_data if board['id'] == 'mb_aster_b550m')

name = f"{test_mobo['brand']} {test_mobo['model']}"
price = '$0000'  # TODO: add price info to synthetic data

card = make_card(name, price, image_placeholder, get_mobo_specs(test_mobo))

display(card)

In [ ]:
test_case = CATALOG["cases"][0]
name = test_case['name']
price = '$0000'  # TODO: add price info to synthetic data

card = make_card(name, price, image_placeholder, get_case_specs(test_case))

display(card)

In [ ]:
# TODO: left off trying to fix the text cutoff in some specs, like front usb in case.